# Samples, the LLN and the CLT, and the math of regression: practice notebook

**Causal Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

This notebook has all the code from the module's web page, plus a few short exercises (cells marked `# YOUR CODE HERE`). It also runs the regressions with `statsmodels`, the package we use in the live sessions.

**In Colab:** *File → Save a copy in Drive* first, so your changes are kept. Run cells with **Shift+Enter**.

## 0. Setup

The Boston housing data (506 census tracts, 1970): `MEDV` = median home value in $1,000s, `RM` = average rooms per home, `LSTAT` = % lower-status residents. The cell below uses a local copy if it finds one, and otherwise downloads the raw file from the UCI repository.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path

SITE = "https://tx58.github.io/causal-analytics/A1/"   # the module's website, e.g. "https://<user>.github.io/<repo>/A1/" (filled in when the site is published)

local = [Path(p) for p in ["boston_small.csv", "data/boston_small.csv", "web/data/boston_small.csv", "boston_housing.csv"]]
found = [p for p in local if p.exists()]
boston = None
if found:
    boston = pd.read_csv(found[0])
elif SITE:
    try:
        boston = pd.read_csv(SITE + "data/boston_small.csv")
    except Exception as err:
        print("website copy not reachable:", err)
if boston is None:
    # the raw data from the UCI repository: no header, columns separated by spaces
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data"
    cols = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
    boston = pd.read_csv(url, sep=r"\s+", header=None, names=cols)
boston = boston[["MEDV", "RM", "LSTAT"]]
print(boston.shape)
boston.describe().round(2)

## Part A. The LLN, the CLT, and a confidence interval

Same code as on the page: a coin with $p = 0.66$, 5,000 sample means of skewed "customer spend", and a 95% interval from one sample of 400.

In [ ]:
rng = np.random.default_rng(1)

# 1. LLN: running share of heads for a coin with p = 0.66
flips = rng.random(10_000) < 0.66
running = flips.cumsum() / np.arange(1, 10_001)
print("share of heads after 10, 100, 10,000 flips:", running[[9, 99, 9_999]].round(3))

# 2. CLT: 5,000 sample means, each from n = 30 skewed draws (customer spend)
n = 30
means = rng.lognormal(3.4, 0.9, size=(5_000, n)).mean(axis=1)
plt.hist(means, bins=50, color="#2563eb")
plt.title(f"5,000 sample means, n = {n}")
plt.xlabel("sample mean ($)")
plt.show()

# 3. A 95% confidence interval from one sample of 400 customers
x = rng.lognormal(3.4, 0.9, size=400)
se = x.std(ddof=1) / np.sqrt(len(x))
print(f"mean {x.mean():.2f}, SE {se:.2f}, 95% CI [{x.mean() - 1.96*se:.2f}, {x.mean() + 1.96*se:.2f}]")

**Exercise A.** Check what "95% confidence" means. The true mean of this spend population is $\mu = e^{3.4 + 0.9^2/2} \approx 44.93$.
Draw 2,000 samples of $n = 100$, build the interval $\bar X \pm 1.96\, s/\sqrt{n}$ for each, and compute the share of intervals that contain $\mu$. Then try $n = 10$ and $n = 1{,}000$. Why is the coverage below 95% for small $n$?

In [ ]:
mu = np.exp(3.4 + 0.9**2 / 2)
# YOUR CODE HERE

## Part B. OLS by hand

The formulas from the derivation: $\hat\beta_1 = \sum (X_i - \bar X)(Y_i - \bar Y) / \sum (X_i - \bar X)^2 = s_{XY}/s_X^2$ and $\hat\beta_0 = \bar Y - \hat\beta_1 \bar X$.

In [ ]:
x, y = boston["RM"].to_numpy(), boston["MEDV"].to_numpy()

# the OLS formulas
b1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
b0 = y.mean() - b1 * x.mean()
print(f"by hand:        b0 = {b0:.3f}, b1 = {b1:.3f}")
print(f"cov / var:      b1 = {np.cov(x, y)[0, 1] / np.var(x, ddof=1):.3f}")
print("numpy polyfit: ", np.polyfit(x, y, 1).round(3))

# what the OLS equations guarantee
e = y - (b0 + b1 * x)
print(f"sum of residuals:     {e.sum():.8f}")
print(f"corr(residuals, RM):  {np.corrcoef(e, x)[0, 1]:.8f}")
print(f"R-squared:            {1 - np.sum(e**2) / np.sum((y - y.mean())**2):.3f}")

The same regression with `statsmodels`: the coefficients match the formulas.

In [ ]:
m = smf.ols("MEDV ~ RM", data=boston).fit()
print(m.summary().tables[1])

**Exercise B.** A regression on a dummy is a difference in means. Create `big = 1` for tracts with more than 6.5 rooms on average (0 otherwise). Regress `MEDV` on `big` and compare the slope and intercept with the two group means.

In [ ]:
# YOUR CODE HERE

## Part C. The standard error of the slope

Classical formula $SER/(\sqrt n\, s_X)$, the heteroskedasticity-robust version, and a check by resampling the tracts.

In [ ]:
x, y = boston["RM"].to_numpy(), boston["MEDV"].to_numpy()
n = len(x)
b1 = np.cov(x, y)[0, 1] / np.var(x, ddof=1)
b0 = y.mean() - b1 * x.mean()
e = y - b0 - b1 * x

ser = np.sqrt(np.sum(e**2) / (n - 2))                 # standard error of the regression
se_classic = ser / (np.sqrt(n) * x.std())             # x.std() divides by n
dx = x - x.mean()                                     # robust (HC1): no equal-variance assumption
se_robust = np.sqrt(np.sum(dx**2 * e**2) / np.sum(dx**2)**2 * n / (n - 2))
print(f"slope {b1:.3f}   classical SE {se_classic:.3f}   robust SE {se_robust:.3f}")
print(f"95% CI (classical): [{b1 - 1.96*se_classic:.2f}, {b1 + 1.96*se_classic:.2f}]")

# how much does the slope really move from sample to sample? resample the tracts
rng = np.random.default_rng(0)
slopes = []
for _ in range(2_000):
    i = rng.integers(0, n, n)                         # draw n tracts with replacement
    slopes.append(np.cov(x[i], y[i])[0, 1] / np.var(x[i], ddof=1))
print(f"SD of the slope across 2,000 resamples: {np.std(slopes):.3f}")

In `statsmodels`, the robust standard error is one option away:

In [ ]:
classic = smf.ols("MEDV ~ RM", data=boston).fit()
robust = smf.ols("MEDV ~ RM", data=boston).fit(cov_type="HC1")
print(f"classical SE {classic.bse['RM']:.3f}, robust SE {robust.bse['RM']:.3f}")

**Exercise C.** The $\sqrt n$ law for a slope. Draw 2,000 random samples of 50 tracts and of 200 tracts (with replacement). For each sample size, compute the SD of the estimated slopes. By how much does it fall when $n$ is four times larger?

In [ ]:
# YOUR CODE HERE

## Part D. Regression anatomy and the OVB formula

In [ ]:
y, x1, x2 = boston["MEDV"].to_numpy(), boston["RM"].to_numpy(), boston["LSTAT"].to_numpy()

# the long regression: MEDV on RM and LSTAT
X = np.column_stack([np.ones_like(x1), x1, x2])
const, b_long, gamma = np.linalg.lstsq(X, y, rcond=None)[0]
print(f"long regression: RM {b_long:.3f}, LSTAT {gamma:.3f}")

# regression anatomy: purge RM of LSTAT, then run a simple regression
fit = np.polyfit(x2, x1, 1)            # RM on LSTAT
x1_tilde = x1 - np.polyval(fit, x2)    # the part of RM that LSTAT can't predict
print(f"Cov(MEDV, RM~) / Var(RM~) = {np.cov(y, x1_tilde)[0, 1] / np.var(x1_tilde, ddof=1):.3f}")

# the OVB formula: short = long + gamma * delta
b_short = np.polyfit(x1, y, 1)[0]
delta = np.polyfit(x1, x2, 1)[0]       # LSTAT on RM
print(f"short {b_short:.3f} = long {b_long:.3f} + ({gamma:.3f}) x ({delta:.3f}) = {b_long + gamma * delta:.3f}")

The same three regressions in `statsmodels`:

In [ ]:
long = smf.ols("MEDV ~ RM + LSTAT", data=boston).fit()
short = smf.ols("MEDV ~ RM", data=boston).fit()
aux = smf.ols("LSTAT ~ RM", data=boston).fit()
print(f"short {short.params['RM']:.3f} = {long.params['RM']:.3f} + {long.params['LSTAT']:.3f} x {aux.params['RM']:.3f}")

**Exercise D.** Turn it around: use regression anatomy to get the coefficient on `LSTAT` in the long regression (purge `LSTAT` of `RM`, then a simple regression of `MEDV` on the residual). Check it against `long.params["LSTAT"]`.

In [ ]:
# YOUR CODE HERE